# Layer 2: 2D Cluster State + GME Witness

**Main result of the project.**

We prepare a 2D cluster state on IQM's **square lattice** and certify **Genuine Multipartite Entanglement (GME)** using only **2 measurement settings** — regardless of system size.

### Why 2D cluster states?
- IQM Emerald *is* a square lattice → CZ gates are native, no SWAP overhead
- Circuit depth = **3 layers** (H + CZ rows + CZ cols), independent of qubit count
- Cluster states are universal resources for measurement-based quantum computing
- → This approach **scales to 1000s of qubits** (scalability bonus criterion)

### GME Witness (Zander et al., Advanced Quantum Technologies 2025)
For a 2-colorable graph state, GME is certified via stabilizer measurements:
$$W = \sum_{i=1}^n \langle g_i \rangle, \quad g_i = X_i \otimes \bigotimes_{j \in N(i)} Z_j$$

- **Biseparable bound**: $W \leq n-1$  
- **Ideal cluster state**: $W = n$  
- $W > n-1$ ⟹ **state is genuinely multipartitely entangled** (proven theorem)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from src.backend import get_backend, get_best_grid, get_calibration_scores

import os
TOKEN = os.environ.get('IQM_TOKEN', None)
DEVICE = 'emerald'
USE_HARDWARE = TOKEN is not None

backend = get_backend(token=TOKEN, device=DEVICE)
print(f'Backend: {backend}  |  Hardware: {USE_HARDWARE}')

## 1. Qubit Selection

Use the best-performing qubits from the device calibration data.

In [ ]:
from src.circuits.cluster_2d import build_cluster_2d_no_measure, checkerboard_coloring
from src.circuits.utils import plot_topology

# Target grid size — start with 4x5=20 qubits, scale up
ROWS, COLS = 4, 5   # → 20 qubits
# ROWS, COLS = 5, 6  # → 30 qubits
# ROWS, COLS = 6, 7  # → 42 qubits
# ROWS, COLS = 6, 9  # → 54 qubits (full Emerald)

if USE_HARDWARE:
    scores = get_calibration_scores(backend)
    layout = get_best_grid(backend, ROWS, COLS, scores=scores)
    print(f'Selected {ROWS}x{COLS} grid:')
    print(f'  Qubits: {layout.qubit_names}')
    print(f'  Avg score: {np.mean(list(layout.scores.values())):.4f}')
    plot_topology(backend, highlight_qubits=layout.qubit_indices,
                  title=f'Selected {ROWS}x{COLS} grid')
else:
    layout = get_best_grid(backend, ROWS, COLS)
    print(f'Simulator: {ROWS}x{COLS} = {ROWS*COLS} qubits')

## 2. Cluster State Circuit (depth 3)

In [ ]:
cluster_circuit = build_cluster_2d_no_measure(ROWS, COLS)

print(f'Circuit stats:')
print(f'  Qubits: {cluster_circuit.num_qubits}')
print(f'  Gates: {cluster_circuit.count_ops()}')
print(f'  Depth (incl. barriers): {cluster_circuit.depth()}')
print(f'  CZ gate depth (excluding barriers): {cluster_circuit.depth(filter_function=lambda x: x.operation.name=="cz")}')
print(f'\nCheckerboard coloring (0=black, 1=white):')
coloring = checkerboard_coloring(ROWS, COLS)
for r in range(ROWS):
    print('  ' + ' '.join('B' if coloring[r*COLS+c]==0 else 'W' for c in range(COLS)))

## 3. GME Witness Measurement

In [ ]:
from src.witnesses.gme_cluster import run_gme, gme_significance, build_gme_circuits

SHOTS = 10000  # max per job on IQM

result = run_gme(backend, cluster_circuit, ROWS, COLS, shots=SHOTS)

n = ROWS * COLS
print(f'=== GME WITNESS RESULTS ({ROWS}x{COLS} = {n} qubits) ===')
print(f'W = {result["W"]:.4f}')
print(f'Biseparable bound (classical): W ≤ {result["biseparable_bound"]}')
print(f'Ideal cluster state:           W = {result["W_ideal"]}')
print(f'Violation: {result["violation"]:.4f}  ({result["significance_sigma"]:.1f}σ)')
print(f'Fraction of ideal: {result["violation_fraction"]*100:.1f}%')
print(f'\nGME CERTIFIED: {result["is_gme"]}')

In [ ]:
# Per-qubit stabilizer values (heatmap)
stab = result['stabilizer_values']
grid = np.array([[stab[r*COLS+c] for c in range(COLS)] for r in range(ROWS)])

fig, ax = plt.subplots(figsize=(COLS*0.9, ROWS*0.9))
im = ax.imshow(grid, vmin=-1, vmax=1, cmap='RdYlGn')
for r in range(ROWS):
    for c in range(COLS):
        ax.text(c, r, f'{grid[r,c]:.2f}', ha='center', va='center', fontsize=8)
plt.colorbar(im, ax=ax, label='Stabilizer expectation ⟨g_i⟩')
ax.set_title(f'Per-qubit stabilizer values — {ROWS}x{COLS} cluster state\n'
             f'W={result["W"]:.3f} > {result["biseparable_bound"]} (GME certified)')
ax.set_xlabel('Column'); ax.set_ylabel('Row')
plt.tight_layout()
plt.savefig('gme_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Scaling: GME vs Qubit Count

In [ ]:
# Sweep grid sizes and measure GME witness
grid_sizes = [(2,3), (3,3), (3,4), (4,4), (4,5), (5,5)]  # adjust based on available qubits
scaling_results = []

for rows, cols in grid_sizes:
    n = rows * cols
    print(f'Testing {rows}x{cols} = {n} qubits...', end=' ')
    circ = build_cluster_2d_no_measure(rows, cols)
    res = run_gme(backend, circ, rows, cols, shots=SHOTS)
    scaling_results.append({
        'n': n, 'rows': rows, 'cols': cols,
        'W': res['W'], 'bound': res['biseparable_bound'],
        'W_ideal': res['W_ideal'],
        'violation': res['violation'],
        'sigma': res['significance_sigma'],
        'is_gme': res['is_gme'],
        'fidelity_approx': res['W'] / res['W_ideal'],
    })
    print(f'W={res["W"]:.2f}/{res["W_ideal"]:.0f}, GME={res["is_gme"]}, {res["significance_sigma"]:.1f}σ')

In [ ]:
ns = [r['n'] for r in scaling_results]
fidelities = [r['fidelity_approx'] for r in scaling_results]
sigmas = [r['sigma'] for r in scaling_results]
gme = [r['is_gme'] for r in scaling_results]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
colors = ['#32a8a4' if g else '#e84545' for g in gme]
ax.bar(ns, fidelities, color=colors, alpha=0.85)
ax.axhline(1.0, color='blue', linestyle='--', label='Ideal')
ax.axhline((np.array(ns)-1)/np.array(ns), color='red', linestyle=':', alpha=0.0)  # placeholder
for i, (n, f, g) in enumerate(zip(ns, fidelities, gme)):
    bound_frac = (n-1)/n
    ax.axhline(bound_frac, color='red', linestyle=':', alpha=0.3)
ax.set_xlabel('Number of qubits')
ax.set_ylabel('W / n  (fraction of ideal)')
ax.set_title('GME Witness Fidelity vs Scale\n(green=GME certified, red=not certified)')
ax.set_ylim(0, 1.1)

ax = axes[1]
ax.semilogy(ns, sigmas, 'o-', color='#32a8a4', linewidth=2, markersize=8)
ax.axhline(3, color='red', linestyle='--', label='3σ threshold')
ax.set_xlabel('Number of qubits')
ax.set_ylabel('Statistical significance (σ)')
ax.set_title('GME Certification Significance')
ax.legend()

plt.suptitle('2D Cluster State GME Witness on IQM Hardware', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('gme_scaling.png', dpi=150, bbox_inches='tight')
plt.show()

max_gme_n = max((r['n'] for r in scaling_results if r['is_gme']), default=0)
print(f'\nMaximum qubits with GME certified: {max_gme_n}')

## 5. Scalability Argument

Our approach scales to **any** size square lattice:
- Circuit depth = **3** (constant, independent of n)
- Measurement settings = **2** (constant, independent of n)
- Classical post-processing = O(n)

This makes it directly applicable to next-generation IQM processors with 100–1000 qubits.

In [ ]:
print('Scalability summary:')
print(f'  Circuit depth:         3 layers (constant)')
print(f'  Measurement settings:  2 (constant)')
print(f'  Native gates used:     H (single-qubit), CZ (native on IQM)')
print(f'  Classical bound:       n-1 (closed form, no optimization needed)')
print(f'  Scales to:             any rows x cols square lattice')
print()
print('Comparison with GHZ approach:')
print(f'  GHZ circuit depth:     n-1  (grows linearly with qubits)')
print(f'  Cluster circuit depth: 3    (constant)')